<a href="https://colab.research.google.com/github/KudjoJedidiah/beginner_projects/blob/main/Loan_Default_Prediction_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import pandas as pd

In [39]:
path='/content/drive/MyDrive/Zindi/LoanDefault/trainperf.csv'
path1='/content/drive/MyDrive/Zindi/LoanDefault/traindemographics.csv'
path2='/content/drive/MyDrive/Zindi/LoanDefault/trainprevloans.csv'
tdemo=pd.read_csv(path1)
perf=pd.read_csv(path)
tprev=pd.read_csv(path2)

In [40]:
perf.head()

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good


In [41]:
perf.value_counts('good_bad_flag')
perf.value_counts('good_bad_flag',normalize=True)

,proportion
good_bad_flag,
Good,0.782051
Bad,0.217949


The target distribution is imbalanced
78% are good loans
22% are bad loans
What this is telling us is that if a model blindly guesses "good" for every single applicant without learning anything,it gets 78% right /

In [42]:
print(f"Number of different customer ids :{perf['customerid'].nunique()}")
print(f"Number of different customer ids :{len(perf['customerid'])}")

Number of different customer ids :4368
Number of different customer ids :4368


Every single row is a unique customer


In [43]:
tdemo.head()

,customerid,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
0,8a858e135cb22031015cbafc76964ebd,1973-10-10 00:00:00.000000,Savings,3.319219,6.528604,GT Bank,NaN,NaN,NaN
1,8a858e275c7ea5ec015c82482d7c3996,1986-01-21 00:00:00.000000,Savings,3.325598,7.119403,Sterling Bank,NaN,Permanent,NaN
2,8a858e5b5bd99460015bdc95cd485634,1987-04-01 00:00:00.000000,Savings,5.746100,5.563174,Fidelity Bank,NaN,NaN,NaN
3,8a858efd5ca70688015cabd1f1e94b55,1991-07-19 00:00:00.000000,Savings,3.362850,6.642485,GT Bank,NaN,Permanent,NaN
4,8a858e785acd3412015acd48f4920d04,1982-11-22 00:00:00.000000,Savings,8.455332,11.971410,GT Bank,NaN,Permanent,NaN


In [44]:
print(f"The total number of rows:{len(tdemo)}")
print(f"The total number of customers:{tdemo['customerid'].nunique()}")


The total number of rows:4346
The total number of customers:4334


In [45]:
print(f"Missing values:\n{tdemo.isnull().sum()}")
print(f"Duplicate values:\n{tdemo.duplicated().sum()}")

Missing values:
customerid                       0
birthdate                        0
bank_account_type                0
longitude_gps                    0
latitude_gps                     0
bank_name_clients                0
bank_branch_clients           4295
employment_status_clients      648
level_of_education_clients    3759
dtype: int64
Duplicate values:
12


In [46]:
tdemo[tdemo.duplicated(subset=['customerid'],keep=False)].sort_values(by='customerid')

,customerid,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
1414,8a858e625c8d993a015c938f829f77ee,1988-12-20 00:00:00.000000,Savings,5.768333,5.561992,First Bank,NaN,Permanent,NaN
1928,8a858e625c8d993a015c938f829f77ee,1988-12-20 00:00:00.000000,Savings,5.768333,5.561992,First Bank,NaN,Permanent,NaN
445,8a858e6c5c88d145015c8b9627cd5a48,1979-09-30 00:00:00.000000,Savings,3.367008,6.497313,Sterling Bank,NaN,Permanent,NaN
1090,8a858e6c5c88d145015c8b9627cd5a48,1979-09-30 00:00:00.000000,Savings,3.367008,6.497313,Sterling Bank,NaN,Permanent,NaN
1996,8a858ec65cc6352b015cc64525ea0763,1985-01-30 00:00:00.000000,Savings,3.845728,7.411737,GT Bank,NaN,Permanent,NaN
1520,8a858ec65cc6352b015cc64525ea0763,1985-01-30 00:00:00.000000,Savings,3.845728,7.411737,GT Bank,NaN,Permanent,NaN
272,8a858edd57f790040157ffe9b6ed3fbb,1988-01-18 00:00:00.000000,Other,3.782563,7.171356,First Bank,NaN,Permanent,Secondary
517,8a858edd57f790040157ffe9b6ed3fbb,1988-01-18 00:00:00.000000,Other,3.782563,7.171356,First Bank,NaN,Permanent,Secondary
4126,8a858f1e5baffcc9015bb02b505f180d,1983-04-06 00:00:00.000000,Savings,6.969350,4.818535,GT Bank,NaN,Permanent,NaN
3021,8a858f1e5baffcc9015bb02b505f180d,1983-04-06 00:00:00.000000,Savings,6.969350,4.818535,GT Bank,NaN,Permanent,NaN


Checking the values in the customerid column which have been duplicated and we can see that they are 100% identical

In [47]:
print(f" Shape before droppind duplicates {tdemo.shape}")

 Shape before droppind duplicates (4346, 9)


In [48]:
uptdemo=tdemo.drop_duplicates(subset=['customerid'],keep='first')

In [49]:
print(f" Shape after dropping duplicates {uptdemo.shape}")

 Shape after dropping duplicates (4334, 9)


We can see that the shape of the dataset has been modified prompting us that the duplicates have been removed

In [50]:
tprev.head()

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,closeddate,referredby,firstduedate,firstrepaiddate
0,8a2a81a74ce8c05d014cfb32a0da1049,301682320,2,2016-08-15 18:22:40.000000,2016-08-15 17:22:32.000000,10000.0,13000.0,30,2016-09-01 16:06:48.000000,NaN,2016-09-14 00:00:00.000000,2016-09-01 15:51:43.000000
1,8a2a81a74ce8c05d014cfb32a0da1049,301883808,9,2017-04-28 18:39:07.000000,2017-04-28 17:38:53.000000,10000.0,13000.0,30,2017-05-28 14:44:49.000000,NaN,2017-05-30 00:00:00.000000,2017-05-26 00:00:00.000000
2,8a2a81a74ce8c05d014cfb32a0da1049,301831714,8,2017-03-05 10:56:25.000000,2017-03-05 09:56:19.000000,20000.0,23800.0,30,2017-04-26 22:18:56.000000,NaN,2017-04-04 00:00:00.000000,2017-04-26 22:03:47.000000
3,8a8588f35438fe12015444567666018e,301861541,5,2017-04-09 18:25:55.000000,2017-04-09 17:25:42.000000,10000.0,11500.0,15,2017-04-24 01:35:52.000000,NaN,2017-04-24 00:00:00.000000,2017-04-24 00:48:43.000000
4,8a85890754145ace015429211b513e16,301941754,2,2017-06-17 09:29:57.000000,2017-06-17 08:29:50.000000,10000.0,11500.0,15,2017-07-14 21:18:43.000000,NaN,2017-07-03 00:00:00.000000,2017-07-14 21:08:35.000000


In [51]:
print(f"The shape of the dataset:{tprev.shape}")
print(f"The number of empty values in each columns:\n{tprev.isnull().sum()}")
print(f"\nThe number of duplicates:{tprev.duplicated().sum()}")
print(f"The number of unique customers:{tprev['customerid'].nunique()}")

The shape of the dataset:(18183, 12)
The number of empty values in each columns:
customerid             0
systemloanid           0
loannumber             0
approveddate           0
creationdate           0
loanamount             0
totaldue               0
termdays               0
closeddate             0
referredby         17157
firstduedate           0
firstrepaiddate        0
dtype: int64

The number of duplicates:0
The number of unique customers:4359


We can see that we have 18183 rows but just 4359 customers that means that one average one customer is to 18183/4359 loans

In [52]:
tprev[['firstduedate','firstrepaiddate']].head(20)

,firstduedate,firstrepaiddate
0,2016-09-14 00:00:00.000000,2016-09-01 15:51:43.000000
1,2017-05-30 00:00:00.000000,2017-05-26 00:00:00.000000
2,2017-04-04 00:00:00.000000,2017-04-26 22:03:47.000000
3,2017-04-24 00:00:00.000000,2017-04-24 00:48:43.000000
4,2017-07-03 00:00:00.000000,2017-07-14 21:08:35.000000
5,2017-04-05 00:00:00.000000,2017-04-04 15:31:47.000000
6,2017-07-04 00:00:00.000000,2017-07-03 23:25:29.000000
7,2017-06-19 00:00:00.000000,2017-06-19 10:00:21.000000
8,2017-07-13 00:00:00.000000,2017-07-10 13:21:53.000000
9,2017-02-21 00:00:00.000000,2017-02-21 05:19:09.000000


In [53]:
tprev['firstduedate']=pd.to_datetime(tprev['firstduedate'])
tprev['firstrepaiddate']=pd.to_datetime(tprev['firstrepaiddate'])

TO CHECK THOSE PEOPLE WHO DIDN'T PAY THEIR LOANS ON TIME

In [54]:
tprev['delay_days']=(tprev['firstrepaiddate']-tprev['firstduedate']).dt.days

In [55]:
tprev['delay_days']

,delay_days
0,-13
1,-4
2,22
3,0
4,11
...,...
18178,-3
18179,-6
18180,-3
18181,19


In [56]:
tprev['delay_days'].describe()

,delay_days
count,18183.000000
mean,-2.473904
std,9.900832
min,-32.000000
25%,-6.000000
50%,-2.000000
75%,0.000000
max,351.000000




*   Some customers repaid up
to 32 days early

*   Half of all loans were settled at least 2 days before the deadline



*   75% of loans were paid on or before the due date.

*   Someone took nearly a full extra year to pay back their loan






In [57]:
summary_df=tprev.groupby('customerid').agg({
    'delay_days':['count','min','max','mean',],
    'loanamount':['min','max','mean','median']
})





*   Count-The total  number of previous loans a customer has taken
*   min-The shortest time a person paid the loan

*   max-The longest delay
*   mean-Their average repayment speed


*   List item
*   List item







In [58]:
print(summary_df.head())
print(f'The shape of the dataset is {summary_df.shape}')

                                 delay_days                   loanamount  \
                                      count min max      mean        min   
customerid                                                                 
8a1088a0484472eb01484669e3ce4e0b          1   6   6  6.000000    10000.0   
8a1a1e7e4f707f8b014f797718316cad          4  -1   1 -0.250000    10000.0   
8a1a32fc49b632520149c3b8fdf85139          7  -2   1 -0.428571    10000.0   
8a1eb5ba49a682300149c3c068b806c7          8 -13   8 -3.125000    10000.0   
8a1edbf14734127f0147356fdb1b1eb2          2  -8   0 -4.000000    10000.0   

                                                                  
                                      max          mean   median  
customerid                                                        
8a1088a0484472eb01484669e3ce4e0b  10000.0  10000.000000  10000.0  
8a1a1e7e4f707f8b014f797718316cad  30000.0  17500.000000  15000.0  
8a1a32fc49b632520149c3b8fdf85139  20000.0  12857.142857 

The column headers have two levels(Multi-index) and ml models cannot read multi-level column headers so they would need to be flattened

In [59]:
summary_df.columns=['-'.join(col).strip() for col in summary_df.columns]
summary_df=summary_df.reset_index()

In [60]:
summary_df.head()

,customerid,delay_days-count,delay_days-min,delay_days-max,delay_days-mean,loanamount-min,loanamount-max,loanamount-mean,loanamount-median
0,8a1088a0484472eb01484669e3ce4e0b,1,6,6,6.000000,10000.0,10000.0,10000.000000,10000.0
1,8a1a1e7e4f707f8b014f797718316cad,4,-1,1,-0.250000,10000.0,30000.0,17500.000000,15000.0
2,8a1a32fc49b632520149c3b8fdf85139,7,-2,1,-0.428571,10000.0,20000.0,12857.142857,10000.0
3,8a1eb5ba49a682300149c3c068b806c7,8,-13,8,-3.125000,10000.0,30000.0,16250.000000,15000.0
4,8a1edbf14734127f0147356fdb1b1eb2,2,-8,0,-4.000000,10000.0,10000.0,10000.000000,10000.0


In [63]:
uptdemo['birthdate']=pd.to_datetime(uptdemo['birthdate'])
uptdemo['age']=2017-uptdemo['birthdate'].dt.year

/tmp/ipykernel_2424/2848861126.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uptdemo['birthdate']=pd.to_datetime(uptdemo['birthdate'])
/tmp/ipykernel_2424/2848861126.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uptdemo['age']=2017-uptdemo['birthdate'].dt.year


In [64]:
print(uptdemo['age'].head())
print(uptdemo['age'].describe())

0    44
1    31
2    30
3    26
4    35
Name: age, dtype: int32
count    4334.000000
mean       32.940009
std         6.138647
min        21.000000
25%        29.000000
50%        32.000000
75%        37.000000
max        56.000000
Name: age, dtype: float64


This gives us the age distribution  of the borrowers

In [67]:
perf['interest']=perf['totaldue']-perf['loanamount']
perf['interest_rate']=(perf['totaldue']-perf['loanamount'])/perf['loanamount']
is_referred=perf['referredby'].notnull().astype(int)

In [71]:
perf['target']=perf['good_bad_flag'].map({'Good':1,'Bad':0})

In [72]:
perf

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag,interest,interest_rate,target
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,4500.0,0.1500,1
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good,2250.0,0.1500,1
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good,2250.0,0.1125,1
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good,1500.0,0.1500,1
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good,4000.0,0.1000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4363,8a858e6d58b0cc520158beeb14b22a5a,302003163,2,2017-07-30 09:19:42.000000,2017-07-30 08:18:30.000000,10000.0,13000.0,30,NaN,Bad,3000.0,0.3000,0
4364,8a858ee85cf400f5015cf44ab1c42d5c,301998967,2,2017-07-27 15:35:47.000000,2017-07-27 14:35:40.000000,10000.0,13000.0,30,NaN,Bad,3000.0,0.3000,0
4365,8a858f365b2547f3015b284597147c94,301995576,3,2017-07-25 16:25:57.000000,2017-07-25 15:24:47.000000,10000.0,11500.0,15,NaN,Bad,1500.0,0.1500,0
4366,8a858f935ca09667015ca0ee3bc63f51,301977679,2,2017-07-14 13:50:27.000000,2017-07-14 12:50:21.000000,10000.0,13000.0,30,8a858eda5c8863ff015c9dead65807bb,Bad,3000.0,0.3000,0


In [73]:
train=pd.merge(perf,summary_df,on='customerid',how='left')
train=pd.merge(train,uptdemo,on='customerid',how='left')

In [74]:
train.shape

(4368, 30)

In [75]:
train.head()

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag,...,loanamount-median,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients,age
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,...,20000.0,1972-01-15,Other,3.432010,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate,45.0
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good,...,NaN,1985-08-23,Savings,3.885298,7.320700,GT Bank,"DUGBE,IBADAN",Permanent,Graduate,32.0
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good,...,10000.0,1984-09-18,Other,11.139350,10.292041,EcoBank,NaN,Permanent,NaN,33.0
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good,...,10000.0,1977-10-10,Savings,3.985770,7.491708,First Bank,NaN,Permanent,NaN,40.0
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good,...,20000.0,1986-09-07,Other,7.457913,9.076574,GT Bank,NaN,Permanent,Primary,31.0


In [77]:
train['target']

,target
0,1
1,1
2,1
3,1
4,1
...,...
4363,0
4364,0
4365,0
4366,0


In [78]:
y=train['target']
drop_cols=[
    'customerid','systemloanid','good_bad_flag','target',
    'birthdate','referredby','approveddate','creationdate','bank_branch_clients'
]
X=train.drop(columns=drop_cols)
X=pd.get_dummies(X,drop_first=True,dummy_na=True)

In [79]:
X.shape

(4368, 48)

In [80]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
import numpy as np

In [84]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
imputer=SimpleImputer(strategy='median')

error_rates=[]

for train_index,test_index in skf.split(X,y):
  X_train_fold,X_test_fold=X.iloc[train_index],X.iloc[test_index]
  y_train_fold,y_test_fold=y.iloc[train_index],y.iloc[test_index]

  X_train_imputed=imputer.fit_transform(X_train_fold)
  X_test_imputed=imputer.transform(X_test_fold)

  model=RandomForestClassifier(n_estimators=100,random_state=42,class_weight='balanced')
  model.fit(X_train_imputed,y_train_fold)

  y_pred=model.predict(X_test_imputed)
  error_rate=1-accuracy_score(y_test_fold,y_pred)
  error_rates.append(error_rate)

print(f"Mean Fold Error Rate: {np.mean(error_rates):.4f}")
print(f"Fold Error Rates:{[round(e,4) for e in error_rates]}")

Mean Fold Error Rate: 0.2228
Fold Error Rates:[0.2231, 0.2243, 0.2174, 0.2199, 0.2291]


In [85]:
importances=pd.Series(model.feature_importances_,index=X.columns)
importances.sort_values(ascending=False).head(10)

,0
delay_days-mean,0.141042
delay_days-max,0.118147
delay_days-min,0.101225
latitude_gps,0.098016
longitude_gps,0.089789
age,0.072978
delay_days-count,0.038301
loannumber,0.038137
loanamount-mean,0.033041
totaldue,0.024542


In [87]:
from sklearn.metrics import confusion_matrix,classification_report
cm=confusion_matrix(y_test_fold,y_pred)
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test_fold,y_pred,target_names=['Bad(0)','Good(1)']))

Confusion Matrix:
[[ 36 154]
 [ 46 637]]

Classification Report:
              precision    recall  f1-score   support

      Bad(0)       0.44      0.19      0.26       190
     Good(1)       0.81      0.93      0.86       683

    accuracy                           0.77       873
   macro avg       0.62      0.56      0.56       873
weighted avg       0.73      0.77      0.73       873



In [92]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import numpy as np

In [90]:
lgb_error_rates=[]

for train_index,test_index in skf.split(X,y):
  X_train_fold,X_test_fold=X.iloc[train_index],X.iloc[test_index]
  y_train_fold,y_test_fold=y.iloc[train_index],y.iloc[test_index]

  lgbm=LGBMClassifier(
     n_estimators=150,
     learning_rate=0.05,
     num_leaves=31,
     random_state=42,
     verbose=-1
  )
  lgbm.fit(X_train_fold,y_train_fold)

  preds=lgbm.predict(X_test_fold)
  error_rate=1-accuracy_score(y_test_fold,preds)
  lgb_error_rates.append(error_rate)

print(f"Mean LightGBM Error Rate: {np.mean(lgb_error_rates):.4f}")
print(f"Fold Error Rates:{[round(e,4) for e in lgb_error_rates]}")

Mean LightGBM Error Rate: 0.2138
Fold Error Rates:[0.2243, 0.2162, 0.214, 0.197, 0.2176]


In [95]:
X['loan_escalation_ratio']=(X['loanamount']/X['loanamount-mean']).fillna(1.0)
X['has_ever_been_late']=(X['delay_days-max']>0).astype(int)
X['totaldue_ratio']=X['totaldue']/X['loanamount']


In [96]:
oof_probs=np.zeros(len(train))


for train_index,test_index in skf.split(X,y):
  X_train_fold,X_test_fold=X.iloc[train_index],X.iloc[test_index]
  y_train_fold,y_test_fold=y.iloc[train_index],y.iloc[test_index]

  lgbm=LGBMClassifier(
     n_estimators=150,
     learning_rate=0.05,
     num_leaves=31,
     random_state=42,
     verbose=-1
  )
  lgbm.fit(X_train_fold,y_train_fold)

  oof_probs[test_index]=lgbm.predict_proba(X_test_fold)[:,1]

best_threshold=0.5
best_error=1.0

for thresh in np.arange(0.2,0.6,0.01):
  preds=(oof_probs>=thresh).astype(int)
  error=1-accuracy_score(y,preds)
  if error<best_error:
    best_error=error
    best_threshold=thresh

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Lowest Out-of-Fold Error Rates:{best_error:.4f}")

Optimal Threshold: 0.46
Lowest Out-of-Fold Error Rates:0.2083


In [98]:
!pip install Catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.3 MB/s eta 0:00:00


In [100]:
from catboost import CatBoostClassifier
oof_lgb=np.zeros(len(train))
oof_cat=np.zeros(len(train))

for train_idx,test_idx in skf.split(X,y):
  X_train,X_test=X.iloc[train_idx],X.iloc[test_idx]
  y_train=y.iloc[train_idx]

  lgbm=LGBMClassifier(
      n_estimators=150,
      learning_rate=0.05,
      num_leaves=31,
      random_state=42,
      verbose=-1
  )
  lgbm.fit(X_train,y_train)
  oof_lgb[test_idx]=lgbm.predict_proba(X_test)[:,1]

  cat=CatBoostClassifier(
      iterations=150,
      learning_rate=0.05,
      random_state=42,
      verbose=0
  )
  cat.fit(X_train_fold,y_train_fold)
  oof_cat[test_idx]=cat.predict_proba(X_test)[:,1]

oof_blend=0.5 * oof_lgb + 0.5 * oof_cat

best_threshold=0.5
best_error=1.0

for thresh in np.arange(0.3,0.65,0.01):
  preds=(oof_blend>=thresh).astype(int)
  error=1-accuracy_score(y,preds)
  if error<best_error:
    best_error=error
    best_threshold=thresh

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Lowest Out-of-Fold Error Rates:{best_error:.4f}")

Optimal Threshold: 0.50
Lowest Out-of-Fold Error Rates:0.1928


In [103]:
test='/content/drive/MyDrive/Zindi/LoanDefault/testperf.csv'
test1='/content/drive/MyDrive/Zindi/LoanDefault/testdemographics.csv'
test2='/content/drive/MyDrive/Zindi/LoanDefault/testprevloans.csv'
test_demo=pd.read_csv(test1)
test_perf=pd.read_csv(test)
test_prev=pd.read_csv(test2)

In [104]:
test_perf.head()

,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby
0,8a858899538ddb8e015390510b321f08,301998974,4,40:48.0,39:35.0,10000,12250.0,30,NaN
1,8a858959537a097401537a4e316e25f7,301963615,10,43:40.0,42:34.0,40000,44000.0,30,NaN
2,8a8589c253ace09b0153af6ba58f1f31,301982236,6,15:11.0,15:04.0,20000,24500.0,30,NaN
3,8a858e095aae82b7015aae86ca1e030b,301971730,8,00:54.0,00:49.0,30000,34500.0,30,NaN
4,8a858e225a28c713015a30db5c48383d,301959177,4,04:33.0,04:27.0,20000,24500.0,30,NaN


In [113]:
# 1. Demographics
test_demo = test_demo.drop_duplicates(subset=['customerid'], keep='first')
test_demo['birthdate'] = pd.to_datetime(test_demo['birthdate'])
test_demo['age'] = 2017 - test_demo['birthdate'].dt.year

# 2. Previous Loans
test_prev['firstduedate'] = pd.to_datetime(test_prev['firstduedate'])
test_prev['firstrepaiddate'] = pd.to_datetime(test_prev['firstrepaiddate'])
test_prev['delay_days'] = (test_prev['firstrepaiddate'] - test_prev['firstduedate']).dt.days

test_summary = test_prev.groupby('customerid').agg({
    'delay_days': ['count', 'min', 'max', 'mean'],
    'loanamount': ['min', 'max', 'mean', 'median']
})
test_summary.columns = ['-'.join(col).strip() for col in test_summary.columns]
test_summary = test_summary.reset_index()

# 3. Current Test Performance
test_perf['interest'] = test_perf['totaldue'] - test_perf['loanamount']
test_perf['interest_rate'] = test_perf['interest'] / test_perf['loanamount']
test_perf['is_referred'] = test_perf['referredby'].notnull().astype(int)

# 4. Merges
test_master = pd.merge(test_perf, test_summary, on='customerid', how='left')
test_master = pd.merge(test_master, test_demo, on='customerid', how='left')

# 5. Ratios
test_master['loan_escalation_ratio'] = (test_master['loanamount'] / test_master['loanamount-mean']).fillna(1.0)
test_master['has_ever_been_late'] = (test_master['delay_days-max'] > 0).astype(int)
test_master['totaldue_ratio'] = test_master['totaldue'] / test_master['loanamount']

# 6. Feature Selection & Column Alignment
drop_cols_test = [
    'customerid', 'systemloanid', 'referredby',
    'approveddate', 'creationdate', 'birthdate', 'bank_branch_clients'
]
X_test = test_master.drop(columns=[c for c in drop_cols_test if c in test_master.columns])
X_test = pd.get_dummies(X_test, drop_first=True, dummy_na=True)

# Strict column alignment against training matrix X
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")

X shape: (4368, 51)
X_test shape: (1450, 51)


In [114]:
test_preds_lgb = np.zeros(len(X_test))
test_preds_cat = np.zeros(len(X_test))

for train_idx, val_idx in skf.split(X, y):
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]

    # 1. LightGBM Fold Model
    lgbm = LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbose=-1
    )
    lgbm.fit(X_train_fold, y_train_fold)
    test_preds_lgb += lgbm.predict_proba(X_test)[:, 1] / skf.n_splits

    # 2. CatBoost Fold Model
    cat = CatBoostClassifier(
        iterations=200,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=0
    )
    cat.fit(X_train_fold, y_train_fold)
    test_preds_cat += cat.predict_proba(X_test)[:, 1] / skf.n_splits

# 50/50 blend of out-of-fold models
final_probs = 0.5 * test_preds_lgb + 0.5 * test_preds_cat

# Apply the optimal 0.50 threshold
final_predictions = (final_probs >= 0.50).astype(int)

# Create submission DataFrame
sub = pd.DataFrame({
    'customerid': test_perf['customerid'],
    'Good_Bad_flag': final_predictions
})

# Save to CSV
sub.to_csv('submission_lgb_cat_blend.csv', index=False)

In [115]:
print(X_test[['delay_days-mean', 'delay_days-max', 'longitude_gps', 'age']].describe())

       delay_days-mean  delay_days-max  longitude_gps         age
count      1442.000000     1442.000000     385.000000  385.000000
mean         -1.998815        3.685853       4.062788   32.618182
std           8.970282       19.035062       7.905697    6.313608
min         -63.000000      -63.000000     -97.883895   21.000000
25%          -5.333333       -2.000000       3.355934   28.000000
50%          -2.200000        0.000000       3.617318   32.000000
75%           0.000000        5.000000       5.985538   36.000000
max         122.000000      367.000000      11.325827   54.000000


In [108]:
sub.head()

,customerid,Good_Bad_flag
0,8a858899538ddb8e015390510b321f08,1
1,8a858959537a097401537a4e316e25f7,1
2,8a8589c253ace09b0153af6ba58f1f31,1
3,8a858e095aae82b7015aae86ca1e030b,1
4,8a858e225a28c713015a30db5c48383d,1


In [116]:
sub['Good_Bad_flag'].value_counts()

,count
Good_Bad_flag,
1,1295
0,155


In [110]:
import pandas as pd

probs_series = pd.Series(final_probs)
print(probs_series.describe())

count    1450.000000
mean        0.851066
std         0.057654
min         0.647162
25%         0.806684
50%         0.863379
75%         0.897275
max         0.971784
dtype: float64


In [111]:
# Check how many non-zero entries exist in key columns
print(X_test[['delay_days-mean', 'delay_days-max', 'longitude_gps', 'age']].describe())

       delay_days-mean  delay_days-max  longitude_gps         age
count           1450.0          1450.0     385.000000  385.000000
mean               0.0             0.0       4.062788   32.618182
std                0.0             0.0       7.905697    6.313608
min                0.0             0.0     -97.883895   21.000000
25%                0.0             0.0       3.355934   28.000000
50%                0.0             0.0       3.617318   32.000000
75%                0.0             0.0       5.985538   36.000000
max                0.0             0.0      11.325827   54.000000
